# Day 014 — Exercise 1: Retrieve with Sources

**Goal:** Implement `retrieve_with_sources` — an upgrade to Day 13's `retrieve_context` that returns source metadata and distance scores alongside each chunk.

**What you build:** A function that queries ChromaDB and returns a list of dicts with keys `text`, `source`, `chunk_index`, and `distance`.

In [ ]:
import ollama
import chromadb

Implement `retrieve_with_sources` below by embedding the question with `ollama.embeddings`, querying the ChromaDB collection, and zipping the returned documents, metadatas, and distances into a list of dicts. Each dict must contain exactly four keys: `text`, `source`, `chunk_index`, and `distance`.

## Your Implementation

In [ ]:
def retrieve_with_sources(
    question: str,
    collection,
    top_k: int = 3,
) -> list[dict]:
    """
    Retrieve top-k chunks with source metadata and distance scores.
    
    Args:
        question:   The user's question to embed and query.
        collection: A ChromaDB Collection object already populated with chunks.
        top_k:      Maximum number of results to return.
    
    Returns:
        List of dicts with keys: text, source, chunk_index, distance.
    """
    # TODO: embed the question using ollama.embeddings(model="nomic-embed-text", ...)
    # TODO: query the collection with query_embeddings=[embedding], n_results=top_k
    # TODO: zip results["documents"][0], results["metadatas"][0], results["distances"][0]
    # TODO: return list of dicts with keys: text, source, chunk_index, distance
    pass

In [ ]:
def _run_checks():
    # Build a tiny in-memory collection for testing
    client = chromadb.Client()
    col_name = "ex01_test"
    try:
        client.delete_collection(col_name)
    except Exception:
        pass
    col = client.create_collection(col_name)
    
    # Add 5 test documents with embeddings
    docs = [
        "Python is a high-level programming language.",
        "ChromaDB is a vector database for embeddings.",
        "Ollama runs large language models locally.",
        "RAG stands for Retrieval-Augmented Generation.",
        "Embeddings represent text as dense vectors.",
    ]
    ids = [f"doc_{i}" for i in range(len(docs))]
    metas = [{"source": f"doc_{i}.txt", "chunk_index": i} for i in range(len(docs))]
    embeddings = [
        ollama.embeddings(model="nomic-embed-text", prompt=d)["embedding"]
        for d in docs
    ]
    col.add(ids=ids, embeddings=embeddings, documents=docs, metadatas=metas)
    
    total = 5
    passed = 0
    
    # Check 1: function returns a list
    try:
        result = retrieve_with_sources("What is Python?", col, top_k=3)
        assert isinstance(result, list), f"Expected list, got {type(result)}"
        passed += 1
        print("✅ Check 1: returns a list")
    except Exception as e:
        print(f"❌ Check 1: returns a list — {e}")
    
    # Check 2: each item is a dict with required keys
    try:
        result = retrieve_with_sources("What is Python?", col, top_k=3)
        required = {"text", "source", "chunk_index", "distance"}
        for item in result:
            assert isinstance(item, dict), f"Item is not a dict: {item}"
            missing = required - set(item.keys())
            assert not missing, f"Missing keys: {missing}"
        passed += 1
        print("✅ Check 2: each item is a dict with keys text/source/chunk_index/distance")
    except Exception as e:
        print(f"❌ Check 2: dict keys — {e}")
    
    # Check 3: honours top_k
    try:
        result2 = retrieve_with_sources("embeddings vectors", col, top_k=2)
        assert len(result2) == 2, f"Expected 2 results for top_k=2, got {len(result2)}"
        passed += 1
        print("✅ Check 3: honours top_k")
    except Exception as e:
        print(f"❌ Check 3: honours top_k — {e}")
    
    # Check 4: text field is a non-empty string
    try:
        result = retrieve_with_sources("language model", col, top_k=3)
        for item in result:
            assert isinstance(item["text"], str) and item["text"].strip(), \
                f"text must be non-empty string, got: {item['text']!r}"
        passed += 1
        print("✅ Check 4: text field is a non-empty string")
    except Exception as e:
        print(f"❌ Check 4: text field — {e}")
    
    # Check 5: distance is a float
    try:
        result = retrieve_with_sources("retrieval generation", col, top_k=3)
        for item in result:
            assert isinstance(item["distance"], float), \
                f"distance must be float, got {type(item['distance'])}"
        passed += 1
        print("✅ Check 5: distance is a float")
    except Exception as e:
        print(f"❌ Check 5: distance type — {e}")
    
    if passed == total:
        print("🎉 Exercise complete!")
    print(f"\nScore: {passed}/{total}")

_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def retrieve_with_sources(
    question: str,
    collection,
    top_k: int = 3,
) -> list[dict]:
    """Retrieve top-k chunks with source metadata and distance scores."""
    q_emb = ollama.embeddings(model="nomic-embed-text", prompt=question)
    results = collection.query(
        query_embeddings=[q_emb["embedding"]],
        n_results=top_k,
    )
    docs      = results["documents"][0]
    metas     = results["metadatas"][0]
    distances = results["distances"][0]
    return [
        {
            "text":        doc,
            "source":      meta["source"],
            "chunk_index": meta["chunk_index"],
            "distance":    dist,
        }
        for doc, meta, dist in zip(docs, metas, distances)
    ]
```

</details>